In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(
    "text",
    data_files="data/train.txt",
    split="train"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B"
)

In [18]:
def tokenize_function(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [12]:
block_size = 32

In [13]:
def group_texts(examples):

    concatenated_examples = {}

    for key in examples.keys():
        concatenated_examples[key] = sum(examples[key], [])

    total_length = len(concatenated_examples["input_ids"])

    total_length = (total_length // block_size) * block_size

    result = {}

    for key, tokens in concatenated_examples.items():

        result[key] = [
            tokens[i:i + block_size]
            for i in range(0, total_length, block_size)
        ]

    result["labels"] = result["input_ids"].copy()

    return result

In [19]:
lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True
)

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [31]:
import pandas as pd

df = pd.DataFrame(
    lm_dataset
)

In [32]:
df.head()

,input_ids,attention_mask,labels
0,"[46208, 4288, 374, 458, 15235, 28383, 448, 264...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[46208, 4288, 374, 458, 15235, 28383, 448, 264..."
1,"[33067, 87858, 42578, 77235, 11, 6915, 2385, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[33067, 87858, 42578, 77235, 11, 6915, 2385, 3..."
2,"[29729, 4583, 15235, 57673, 11, 2670, 7299, 41...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[29729, 4583, 15235, 57673, 11, 2670, 7299, 41..."
3,"[10603, 60666, 87468, 13, 220, 1519, 20699, 19...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[10603, 60666, 87468, 13, 220, 1519, 20699, 19..."
4,"[3635, 656, 36925, 776, 4405, 13, 220, 1986, 4...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[3635, 656, 36925, 776, 4405, 13, 220, 1986, 4..."
